# Sanity Check - Step 07: Epoching

Überprüft:
- Epochs erfolgreich erstellt
- Event-Anzahl und -Typen
- Epoch-Größe (Anzahl und Dimensionen)
- Baseline-Korrektur

In [ ]:
import os
import sys
from pathlib import Path
import mne
import numpy as np

root = Path.cwd()
candidate_roots = [root, root.parent, root.parent.parent]
for candidate in candidate_roots:
    pipeline_dir = candidate / "eeg_pipeline"
    if pipeline_dir.exists() and str(pipeline_dir) not in sys.path:
        sys.path.insert(0, str(pipeline_dir))
    if candidate.exists() and str(candidate) not in sys.path:
        sys.path.insert(0, str(candidate))

try:
    from eeg_pipeline import config
except ModuleNotFoundError:
    import config

print("Setup erfolgreich")

In [ ]:
# Manuelle Auswahl fuer diesen Notebook-Run
subject_id = "01"  # z.B. "02"
person = "P1"  # "P1" oder "P2"
if person not in {"P1", "P2"}:
    raise ValueError("person muss P1 oder P2 sein")
os.environ["EEG_SUBJECT"] = subject_id
os.environ["EEG_PERSON"] = person
print(f"Manuell gesetzt: EEG_SUBJECT={os.environ['EEG_SUBJECT']}, EEG_PERSON={os.environ['EEG_PERSON']}")

## 1. Epochs laden

In [ ]:
subject_id = os.getenv("EEG_SUBJECT", config.SUBJECTS[0]).strip()
person = os.getenv("EEG_PERSON", "P1").strip().upper()
if person not in {"P1", "P2"}:
    raise ValueError("EEG_PERSON muss P1 oder P2 sein")

epoch_path = config.OUTPUT_DIR / f"sub-{subject_id}_{person}_epoch.fif"

if epoch_path.exists():
    epochs = mne.read_epochs(str(epoch_path), preload=False)
    print(f"✓ Epochs loaded for {person}")
else:
    print(f"✗ Epoch file not found")

## 2. Epoch-Grundlagen

In [ ]:
print(f"=== EPOCH DETAILS ===")
print(f"Number of epochs: {len(epochs)}")
print(f"Number of channels: {len(epochs.ch_names)}")
print(f"Samples per epoch: {epochs.get_data().shape[2]}")
print(f"Sampling rate: {epochs.info['sfreq']} Hz")

print(f"\nEvent types: {epochs.event_id}")
print(f"Time window: [{epochs.times[0]:.3f}, {epochs.times[-1]:.3f}] s")
print(f"Expected duration: {config.EPOCH_TMAX - config.EPOCH_TMIN:.3f}s")
print(f"Actual duration: {epochs.times[-1] - epochs.times[0]:.3f}s")

## 3. Baseline & Data Quality

In [ ]:
print(f"=== DATA QUALITY ===")
if epochs.baseline is not None:
    print(f"✓ Baseline applied: {epochs.baseline}")
else:
    print(f"⚠ No baseline correction")

data = epochs.get_data()
nan_count = int(np.isnan(data).sum())
inf_count = int(np.isinf(data).sum())

if nan_count == 0 and inf_count == 0:
    print(f"✓ No NaN or Inf values")
else:
    print(f"✗ Found {nan_count} NaN and {inf_count} Inf values")

bads = epochs.info.get('bads', [])
if len(bads) == 0:
    print(f"✓ No bad channels marked")
else:
    print(f"⚠ Bad channels marked: {len(bads)}")

## 4. Event-Verteilung

In [ ]:
if len(epochs.event_id) > 0:
    print(f"Event Distribution:")
    event_counts = epochs.event_id
    # Create a summary
    for event_name in epochs.event_id.keys():
        try:
            count = len(epochs[event_name])
            print(f"  {event_name}: {count} epochs")
        except:
            print(f"  {event_name}: (could not count)")
else:
    print(f"No events found in epochs")